# Non-Separable Time and Graph Gaussian Processes with a Kronecker-Sum HSGP

In many forecasting problems the data come as a panel: one weekly series per region, for many regions. The regions share a slow national trend. The regional deviations from that trend are not independent: neighboring regions move together, and rough spatial patterns (one region up, its neighbor down) do not last as long as smooth ones. We want a prior on the latent panel that encodes these three facts and that is cheap enough to run inside a full Bayesian workflow.

The Hilbert space Gaussian process (HSGP) approximation gives us a fixed basis and diagonal weights for a stationary kernel on an interval. If you are new to HSGPs, start with [A Conceptual and Practical Introduction to Hilbert Space GPs Approximation Methods](https://juanitorduz.github.io/hsgp_intro/); this notebook builds on it. Here we extend the construction to the product space *interval times graph*. The eigenfunctions of the product space are products of the eigenfunctions of the two factors, $\phi_j(t)\, u_v(r)$, and the corresponding eigenvalues are sums, $\lambda_j + \mu_v$. We evaluate the Matérn spectral density at these sums. The result is a non-separable spatio-temporal kernel, and the HSGP computational structure survives: a hyperparameter-free basis, diagonal weights, and two matrix multiplications.

Concretely, we do the following:

- We derive the basis on the time box and the eigenbasis of the graph Laplacian, and we combine them into three priors that share the same hyperparameters: independent regions, a separable (product) kernel, and the Kronecker-sum kernel.
- We simulate a weekly panel for the 16 German states with a data generating process that is deliberately *not* the model.
- We fit the three models with NUTS in [NumPyro](https://num.pyro.ai/), forecast 13 weeks ahead with [`numpyro-forecast`](https://github.com/juanitorduz/numpyro_forecast), and compare the forecasts with the CRPS.
- We run a rolling-origin backtest to check that the ranking is not an artifact of a single split.

Related material: the PyMC example [HSGP Advanced Usage, example 2](https://www.pymc.io/projects/examples/en/latest/gaussian_processes/HSGP-Advanced.html#example-2-an-hsgp-that-exploits-kronecker-structure) builds an HSGP with Kronecker *product* structure (a separable kernel); the [NYC BYM example](https://www.pymc.io/projects/examples/en/latest/spatial/nyc_bym.html) uses a graph to model spatial proximity with a conditional autoregressive prior; the post [Laplacian Eigenmaps](https://juanitorduz.github.io/laplacian_eigenmaps_dim_red/) and the [PyData Berlin 2018 slides](https://juanitorduz.github.io/documents/orduz_pydata2018.pdf) cover the graph Laplacian and its spectrum.

**Remark.** The gain of the non-separable kernel over the separable one is modest and depends on the data. The value of this notebook is the construction itself, and the checks that go with it: normalization, node standardization, basis size, and the time box.

## Prepare Notebook

In [ ]:
import time
from typing import Literal

import arviz as az
import geopandas as gpd
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import numpyro
import numpyro.distributions as dist
import polars as pl
import preliz as pz
from jax import random
from jaxtyping import Array, Float
from numpyro.contrib.hsgp.approximation import hsgp_matern, linear_approximation
from numpyro.contrib.hsgp.laplacian import eigenfunctions, sqrt_eigenvalues
from numpyro.contrib.hsgp.spectral_densities import diag_spectral_density_matern
from numpyro.handlers import do, substitute
from numpyro.infer import MCMC, NUTS, Predictive, init_to_median
from numpyro_forecast import (
    Horizon,
    backtest,
    eval_coverage,
    eval_crps,
    eval_rmse,
    forecast,
    predict,
)
from numpyro_forecast.features import fourier_features

numpyro.set_host_device_count(n=4)

seed: int = 42
rng_key = random.PRNGKey(seed=seed)

HDI_PROB = 0.94

az.style.use("arviz-darkgrid")
az.rcParams["stats.ci_kind"] = "hdi"
az.rcParams["stats.ci_prob"] = HDI_PROB
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"

%config InlineBackend.figure_format = "retina"

## The Problem and the Estimand

We observe a panel $y_{t, r}$ for weeks $t = 1, \ldots, T$ and regions $r = 1, \ldots, R$. The regions are the nodes of an undirected graph with adjacency matrix $A$: two regions are adjacent if they share a border. The graph is our notion of spatial distance, in the same way as the [NYC BYM example](https://www.pymc.io/projects/examples/en/latest/spatial/nyc_bym.html) uses the adjacency of census tracts. We want to forecast $H$ weeks ahead for every region.

The estimand is the predictive distribution $\text{P}(y_{T + h, r} \mid y_{1:T, 1:R})$ for $h = 1, \ldots, H$ and every region $r$. As mentioned in the introduction, we will compare three models (details below). All models share the same observation equation. Let $f(t, r)$ be a latent Gaussian process on the product space, $a_r$ a regional intercept, $s(t)$ a yearly seasonal term, and $\sigma$ the observation scal. We then consider a panel-level model of the form:

$$
y_{t, r} = a_r + f(t, r) + s(t) + \varepsilon_{t, r}, \qquad \varepsilon_{t, r} \sim \text{StudentT}(\nu_{\varepsilon}, 0, \sigma).
$$

The three models differ only in the prior on $f$.

We evaluate the model performance using the continuous ranked probability score (CRPS).

## From the Interval to the Product Space

This section derives the construction step by step. We start with the HSGP approximation on an interval, then the graph Laplacian, and then we combine the two. For more details, see [A Conceptual and Practical Introduction to Hilbert Space GPs Approximation Methods](https://juanitorduz.github.io/hsgp_intro/). Here we do a fast walkthrough to stablish notation.

### Stationary Kernels and Spectral Densities

A kernel $k(t, t')$ is stationary if it only depends on the difference $\tau = t - t'$. By Bochner's theorem, a stationary kernel is the Fourier transform of a positive function $S(\omega)$, its spectral density:

$$
k(\tau) = \int_{-\infty}^{\infty} S(\omega)\, e^{i \omega \tau}\, d\omega .
$$

For the Matérn family with smoothness $\nu$, lengthscale $\ell$, and variance $\sigma_f^2$, the spectral density in one dimension is ([Rasmussen and Williams, 2006, chapter 4](https://gaussianprocess.org/gpml/chapters/RW4.pdf), equation 4.15)

$$
S(\omega) = \sigma_f^2\, \frac{2 \sqrt{\pi}\, \Gamma(\nu + 1/2)\, (2\nu)^{\nu}}{\Gamma(\nu)\, \ell^{2\nu}} \left(\frac{2\nu}{\ell^2} + \omega^2\right)^{-(\nu + 1/2)} .
$$

We write $\kappa^2 = 2\nu / \ell^2$. The shape of the density is then $(\kappa^2 + \omega^2)^{-(\nu + 1/2)}$ up to a constant: a Matérn process has most of its power at frequencies below $\kappa$, and the power decays polynomially above it. A longer lengthscale means a smaller $\kappa$ and less power at high frequencies.

### HSGP on an Interval

The HSGP approximation ([Solin and Särkkä, 2020](https://link.springer.com/article/10.1007/s11222-019-09886-w); [Riutort-Mayol et al., 2023](https://link.springer.com/article/10.1007/s11222-022-10167-2)) uses that a stationary kernel is a function of the Laplacian, $K = S(\sqrt{-\Delta})$, and that on the box $[-L, L]$ with Dirichlet boundary conditions the Laplacian has a closed-form eigenbasis. Let $m$ be the number of basis functions, $j = 1, \ldots, m$, and $\omega_j = j \pi / (2L)$. The eigenfunctions and eigenvalues are

$$
\phi_j(t) = \frac{1}{\sqrt{L}} \sin\left(\omega_j (t + L)\right), \qquad \lambda_j = \omega_j^2 .
$$

The approximation replaces the kernel by its expansion in this basis, with the spectral density evaluated at the square root of the eigenvalues:

$$
k(t, t') \approx \sum_{j = 1}^{m} S\!\left(\sqrt{\lambda_j}\right) \phi_j(t)\, \phi_j(t') .
$$

A Gaussian process with this kernel is a linear combination of the basis functions with independent coefficients,

$$
f(t) = \sum_{j = 1}^{m} \sqrt{S\!\left(\sqrt{\lambda_j}\right)}\, \beta_j\, \phi_j(t), \qquad \beta_j \sim \text{Normal}(0, 1),
$$

which is a linear model with $m$ coefficients. The basis does not depend on the kernel hyperparameters; only the weights $S(\sqrt{\lambda_j})$ do. For forecasting, the box must contain the training window *and* the forecast horizon. With $D = T + H$ time steps we center the grid and set $L = c\, (D - 1) / 2$ with $c > 1$ (we use $c = 1.5$).

We look at the three ingredients: the basis functions, the weights, and the quality of the approximation.

In [ ]:
T_OBS = 156
H = 13
DURATION = T_OBS + H
PERIOD = 52.0

M_BASIS = 48
C_BOX = 1.5
NU = 1.5

t_grid = jnp.arange(DURATION, dtype=jnp.float32)
t_centered = t_grid - (DURATION - 1) / 2.0
ell_box = float(C_BOX * (DURATION - 1) / 2.0)

phi_full = eigenfunctions(t_centered, ell=ell_box, m=M_BASIS)  # (DURATION, m)

print(f"box half-width L = {ell_box:.1f} weeks, basis shape = {phi_full.shape}")

In [ ]:
fig, ax = plt.subplots()
for j in range(4):
    ax.plot(
        np.asarray(t_grid), np.asarray(phi_full[:, j]), label=f"$\\phi_{{{j + 1}}}$"
    )
ax.axvline(x=T_OBS, color="black", linestyle="--", label="end of training data")
ax.set(xlabel="week", ylabel="$\\phi_j(t)$")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=5)
fig.suptitle(
    "The first four HSGP basis functions on the box", fontsize=18, fontweight="bold"
);

The basis functions are sine waves on the box. The box extends beyond the data on both sides (the grid is centered), which keeps the Dirichlet boundary away from the forecast horizon. The weights $S(\sqrt{\lambda_j})$ decide how much each frequency contributes. We evaluate the Matérn-$3/2$ spectral density (the `numpyro` implementation of the formula above) at the eigenvalues for three lengthscales.

In [ ]:
fig, ax = plt.subplots()
for length in [10.0, 20.0, 40.0]:
    s_j = diag_spectral_density_matern(
        nu=NU, alpha=1.0, length=length, ell=ell_box, m=M_BASIS, dim=1
    )
    ax.plot(
        np.arange(1, M_BASIS + 1),
        np.asarray(s_j),
        label=f"$\\ell$ = {length:.0f}",
    )
ax.set(xlabel="basis function $j$", ylabel="$S(\\sqrt{\\lambda_j})$", yscale="log")
ax.legend()
fig.suptitle(
    "Matérn-3/2 spectral weights for three lengthscales", fontsize=18, fontweight="bold"
);

A longer lengthscale concentrates the weight on the first few basis functions. This is also why the basis can be truncated: for $\ell = 20$ the weight of $j = 48$ is four orders of magnitude below the weight of $j = 1$. We now compare the approximate kernel $\sum_j S(\sqrt{\lambda_j}) \phi_j(t) \phi_j(t')$ with the exact Matérn-$3/2$ kernel $k(\tau) = \sigma_f^2 (1 + \sqrt{3} |\tau| / \ell) \exp(-\sqrt{3} |\tau| / \ell)$, as a function of $t$ for a fixed $t'$ in the middle of the data.

In [ ]:
def matern32_kernel(tau: Array, length: float, sigma_f: float = 1.0) -> Array:
    "Exact Matérn-3/2 kernel as a function of the time difference tau."
    d = jnp.sqrt(3.0) * jnp.abs(tau) / length
    return sigma_f**2 * (1.0 + d) * jnp.exp(-d)


t_ref = T_OBS // 2
fig, ax = plt.subplots()
for i, length in enumerate([10.0, 20.0, 40.0]):
    s_j = diag_spectral_density_matern(
        nu=NU, alpha=1.0, length=length, ell=ell_box, m=M_BASIS, dim=1
    )
    k_hsgp = (phi_full * s_j) @ phi_full[t_ref]  # sum_j S_j phi_j(t) phi_j(t_ref)
    k_exact = matern32_kernel(t_grid - t_grid[t_ref], length)
    ax.plot(
        np.asarray(t_grid),
        np.asarray(k_exact),
        color=f"C{i}",
        label=f"exact, $\\ell$ = {length:.0f}",
    )
    ax.plot(
        np.asarray(t_grid),
        np.asarray(k_hsgp),
        color=f"C{i}",
        linestyle="--",
        label=f"HSGP, $\\ell$ = {length:.0f}",
    )
ax.set(xlabel="week $t$", ylabel="$k(t, t')$")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
fig.suptitle(
    f"HSGP approximation of the Matérn-3/2 kernel (m = {M_BASIS})",
    fontsize=18,
    fontweight="bold",
);

The approximation is accurate for these lengthscales and $m = 48$. It degrades when $\ell$ is short relative to the box, because the density then has power beyond the last basis frequency $\omega_m$. We come back to this in the section on the basis size.

### The Graph Laplacian

Let $A$ be the adjacency matrix of the graph and $D_G$ the diagonal matrix of node degrees $d_r = \sum_{r'} A_{r r'}$. The graph Laplacian is $L_G = D_G - A$. For any vector $x$ on the nodes,

$$
x^\top L_G x = \frac{1}{2} \sum_{r, r'} A_{r r'} (x_r - x_{r'})^2 ,
$$

which is the sum of squared differences across the edges. So $x^\top L_G x$ measures how *rough* $x$ is over the graph, in the same way as $\int (dx / dt)^2 dt$ measures how rough a function is on an interval. See [Laplacian Eigenmaps](https://juanitorduz.github.io/laplacian_eigenmaps_dim_red/) and the [PyData Berlin 2018 slides](https://juanitorduz.github.io/documents/orduz_pydata2018.pdf) for a longer discussion.

$L_G$ is symmetric and positive semidefinite, so it has an orthonormal eigendecomposition $L_G = U \operatorname{diag}(\mu) U^\top$ with eigenvalues $0 = \mu_0 \le \mu_1 \le \cdots \le \mu_{R - 1}$ and eigenvectors $u_v$ (the columns of $U$). The eigenvalue is the roughness of its eigenvector, $\mu_v = u_v^\top L_G u_v$. The first eigenvector $u_0$ is constant. Eigenvectors with larger $\mu_v$ oscillate faster over the graph, in the same sense that $\sin(\omega_j t)$ oscillates faster for larger $j$. This makes the pair $(\mu_v, u_v)$ the graph analogue of $(\lambda_j, \phi_j)$.

**Remark** The quantity $\mu_0 = 0$ will represent the *national average* mode in the example below.


We want to apply this contruction to our both Laplacians above to construct the corresponding HSGP approximation on the product, see [Gaussian Processes: HSGP Advanced Usage, An HSGP that exploits Kronecker structure](https://www.pymc.io/projects/examples/en/latest/gaussian_processes/HSGP-Advanced.html#example-2-an-hsgp-that-exploits-kronecker-structure). A Matérn Gaussian process on a graph ([Borovitskiy et al., 2021](https://proceedings.mlr.press/v130/borovitskiy21a.html)) follows the same recipe as on the interval: a spectral density evaluated at the Laplacian eigenvalues, $K_G = \sum_v S(\sqrt{\mu_v})\, u_v u_v^\top$.

### The Kronecker Sum

The Kronecker sum of two operators $P$ (on the interval) and $Q$ (on the graph) is $P \oplus Q = P \otimes I + I \otimes Q$. If $P \phi_j = \lambda_j \phi_j$ and $Q u_v = \mu_v u_v$, then the product function $\phi_j(t) u_v(r)$ is an eigenfunction of the sum with eigenvalue $\lambda_j + \mu_v$:

$$
(P \oplus Q)\, (\phi_j \otimes u_v) = (P \phi_j) \otimes u_v + \phi_j \otimes (Q u_v) = (\lambda_j + \mu_v)\, (\phi_j \otimes u_v) .
$$

On the product space *interval times graph* the natural Laplacian is the Kronecker sum $-\Delta_t \oplus L_G$, because the Laplacian of a product space is the sum of the Laplacians of the factors (as $\partial_x^2 + \partial_y^2$ on the plane). Its eigenpairs are

$$
\left(\lambda_j + \mu_v,\; \phi_j(t)\, u_v(r)\right), \qquad j = 1, \ldots, m, \quad v = 0, \ldots, R - 1 .
$$

Now we evaluate the Matérn spectral density at the sum. The graph eigenvalues have no units, while $\lambda_j$ has units of inverse squared time, so we need a conversion factor: we use $\kappa^2 \rho$ with $\rho \ge 0$ dimensionless. Plugging $\omega^2 = \lambda_j + \kappa^2 \rho\, \mu_v$ into $(\kappa^2 + \omega^2)^{-(\nu + 1/2)}$ and pulling out $\kappa^2$ gives the spectral weights

$$
W_{j v} = C \left(\kappa^2 + \lambda_j + \kappa^2 \rho\, \mu_v\right)^{-(\nu + 1/2)}
        = C' \left(1 + \frac{\lambda_j}{\kappa^2} + \rho\, \mu_v\right)^{-(\nu + 1/2)} .
$$

The constant $C'$ is fixed by the normalization below. The process is

$$
f(t, r) = \sum_{j, v} \sqrt{W_{j v}}\, \beta_{j v}\, \phi_j(t)\, u_v(r), \qquad \beta_{j v} \sim \text{Normal}(0, 1).
$$

In matrix form, with $\Phi$ the $T \times m$ matrix of time basis functions and $B$ the $m \times R$ matrix of coefficients, $F = \Phi\, (\sqrt{W} \circ B)\, U^\top$: two matrix multiplications.

**Interpretation.** Fix a graph mode $v$ and look at the weights as a function of $j$ only. Since $\lambda_j / \kappa^2 + (1 + \rho \mu_v) = (1 + \rho \mu_v) \left(1 + \lambda_j / (\kappa^2 (1 + \rho \mu_v))\right)$, they are the weights of a one-dimensional Matérn-$\nu$ process in time with inverse squared lengthscale $\kappa^2 (1 + \rho \mu_v)$, times a mode-specific constant. Comparing with the Matérn spectral density above, graph mode $v$ is a temporal Matérn-$\nu$ HSGP with its own lengthscale and amplitude,

$$
\ell_v = \frac{\ell}{\sqrt{1 + \rho\, \mu_v}}, \qquad \alpha_v = (1 + \rho\, \mu_v)^{-\nu} ,
$$

so that $W_{jv} \propto S(\sqrt{\lambda_j};\, \alpha_v, \ell_v)$ with $S$ the one-dimensional Matérn density. This is also how we compute the weights in code: we reuse the Matérn spectral density of `numpyro.contrib.hsgp` and only supply the pair $(\alpha_v, \ell_v)$ for every graph mode.

Three consequences follow:

- $v = 0$ is the shared national trend, with the longest lengthscale $\ell$ and the largest variance.
- Rough spatial patterns (large $\mu_v$) have shorter memory and smaller amplitude.
- $\rho \to 0$ makes all modes identical (independent regions with shared hyperparameters); $\rho \to \infty$ leaves only $v = 0$ (complete pooling).

So $\rho$ is a single *spatial pooling strength*, and the graph decides which deviations are suppressed first. For a forecast this means that regional forecasts shrink toward the national forecast at a rate that increases with the horizon and with spatial roughness.

### Three Priors with the Same Hyperparameters

To attribute a difference in forecast quality to non-separability alone, we compare three priors that share $(\sigma_f, \ell, \rho)$ and differ only in how the two spectra are combined:

| mode | $W_{jv}$ | $\ell_v$ | $\alpha_v$ | kernel |
|---|---|---|---|---|
| `kron_sum` | $(1 + \lambda_j / \kappa^2 + \rho \mu_v)^{-(\nu + 1/2)}$ | $\ell / \sqrt{1 + \rho \mu_v}$ | $(1 + \rho \mu_v)^{-\nu}$ | non-separable |
| `separable` | $(1 + \lambda_j / \kappa^2)^{-(\nu + 1/2)}\, (1 + \rho \mu_v)^{-(\nu + 1/2)}$ | $\ell$ | $(1 + \rho \mu_v)^{-(\nu + 1/2)}$ | Matérn(time) $\otimes$ graph-Matérn |
| `independent` | $(1 + \lambda_j / \kappa^2)^{-(\nu + 1/2)}$ with $U = I$ | $\ell$ | $1$ | one HSGP per region |

The last two columns say the same thing as $W_{jv}$: every prior is a collection of $R$ temporal Matérn HSGPs, one per graph mode, and the graph eigenvalue $\mu_v$ sets the amplitude and (for the Kronecker sum only) the lengthscale of mode $v$.

The separable kernel is the product of a temporal Matérn kernel and a graph Matérn kernel (an intrinsic coregionalization model with a fixed coregionalization matrix). This is the structure of the PyMC [HSGP Kronecker example](https://www.pymc.io/projects/examples/en/latest/gaussian_processes/HSGP-Advanced.html#example-2-an-hsgp-that-exploits-kronecker-structure): a product of kernels has a Kronecker *product* covariance and its spectrum is the product of the two spectra. In the separable kernel every graph mode has the *same* temporal lengthscale; the graph only changes the amplitude of each mode. In the Kronecker sum the spectra are combined *inside* the density, and the graph changes both the amplitude and the lengthscale.

### Normalization and Node Standardization

We fix the overall scale by trace normalization. Since $\sum_r u_v(r)^2 = 1$ and the average of $\phi_j(t)^2$ over the box is $1 / (2L)$, the average prior variance over time and regions is $\sum_{j v} W_{j v} / (2 L R)$. We choose $C'$ so that this average equals $\sigma_f^2$.

Graph Matérn priors have a marginal variance that depends on the node: on the German adjacency graph the prior variance at Bremen (degree 1) and at Lower Saxony (degree 9) differ by a factor of about four. This is an accident of the adjacency, not a modeling statement. Let $v_r = \sum_{j v} W_{j v}\, u_v(r)^2 / (2L)$ be the prior variance at region $r$. We rescale $f(\cdot, r)$ by $s_r = \sigma_f / \sqrt{v_r}$. With $D = \operatorname{diag}(s)$ the kernel becomes $D K D$, which is still positive semidefinite and still two matrix multiplications (we replace $U$ by $D U$). Every region then has prior variance $\sigma_f^2$ and the graph only shapes the *correlations*.

### Basis Size

The number of time basis functions $m$ must resolve the *shortest* effective lengthscale, $\ell_{\min} = \ell / \sqrt{1 + \rho\, \mu_{\max}}$, not $\ell$. The rule of thumb of Riutort-Mayol et al. for a Matérn-$3/2$ kernel becomes $m \gtrsim 1.75\, c\, (D / 2) / \ell_{\min}$. If $m$ is too small the truncation silently over-smooths the rough spatial modes, which acts like a larger $\rho$. We check this rule against the posterior later in the notebook.

## The Graph

We use the adjacency graph of the 16 German states (Bundesländer). Two states are adjacent if their polygons share a border. We read the state polygons from a GeoJSON file ([isellsoap/deutschlandGeoJSON](https://github.com/isellsoap/deutschlandGeoJSON), low-resolution version, derived from data of the Federal Agency for Cartography and Geodesy under the [dl-de/by-2-0](https://www.govdata.de/dl-de/by-2-0) license) and derive the adjacency from the geometry, so nothing is typed by hand.

In [ ]:
states_gdf = gpd.read_file("../data/germany_states.geojson")
states_gdf["code"] = states_gdf["id"].str.replace("DE-", "")
states_gdf = states_gdf.set_index("code")

STATES = list(states_gdf.index)
R = len(STATES)
state_index = {s: i for i, s in enumerate(STATES)}

geometries = states_gdf.geometry.to_numpy()
adjacency = np.zeros((R, R))
for i in range(R):
    for j in range(R):
        if i != j and geometries[i].intersects(geometries[j]):
            adjacency[i, j] = 1.0

EDGES = [
    (STATES[i], STATES[j]) for i in range(R) for j in range(i + 1, R) if adjacency[i, j]
]
positions = {
    s: (p.x, p.y) for s, p in states_gdf.geometry.representative_point().items()
}
degree = adjacency.sum(axis=1)

print(f"{R} states, {len(EDGES)} borders")

pl.DataFrame({"state": STATES, "name": states_gdf["name"].to_list(), "degree": degree})

In [ ]:
graph = nx.Graph(EDGES)

fig, ax = plt.subplots(figsize=(9, 10))
states_gdf.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=1)
nodes = nx.draw_networkx_nodes(
    graph,
    pos=positions,
    nodelist=STATES,
    node_color=degree,
    cmap="viridis",
    node_size=700,
    ax=ax,
)
nx.draw_networkx_edges(graph, pos=positions, edge_color="C1", width=1.5, ax=ax)
nx.draw_networkx_labels(
    graph, pos=positions, font_color="white", font_weight="bold", ax=ax
)
fig.colorbar(nodes, ax=ax, label="degree", shrink=0.6)
ax.set(xlabel="longitude", ylabel="latitude")
fig.suptitle("Adjacency graph of the German states", fontsize=18, fontweight="bold");

The graph is irregular: Lower Saxony (NI) has nine neighbors, Bremen (HB) has one. We compute the Laplacian and its eigendecomposition once, outside any model, in JAX. We clip tiny negative eigenvalues from round-off and set $\mu_0 = 0$ exactly.

In [ ]:
def graph_laplacian(adjacency: Float[Array, "R R"]) -> Float[Array, "R R"]:
    "Combinatorial graph Laplacian L = D - A of a symmetric adjacency matrix."
    if not jnp.allclose(adjacency, adjacency.T):
        raise ValueError("adjacency must be symmetric")
    return jnp.diag(adjacency.sum(axis=1)) - adjacency


def laplacian_eigenpairs(
    adjacency: Float[Array, "R R"],
) -> tuple[Float[Array, " R"], Float[Array, "R R"]]:
    "Ascending eigenvalues mu (mu[0] = 0) and eigenvectors U (columns)."
    mu, U = jnp.linalg.eigh(graph_laplacian(adjacency))
    mu = jnp.clip(mu, 0.0, None).at[0].set(0.0)
    return mu, U


mu, U = laplacian_eigenpairs(jnp.asarray(adjacency))

fig, ax = plt.subplots()
ax.stem(np.arange(R), np.asarray(mu), basefmt=" ")
ax.set(xlabel="graph mode $v$", ylabel="$\\mu_v$", xticks=np.arange(R))
fig.suptitle("Eigenvalues of the graph Laplacian", fontsize=18, fontweight="bold");

The first eigenvector is constant. The second one (the Fiedler vector) splits the graph into two smooth halves, south-west against north-east. Higher modes oscillate faster over the graph.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 9), layout="constrained")
vmax = float(jnp.abs(U).max())
for v, ax in enumerate(axes.flat):
    states_gdf.plot(ax=ax, color="lightgray", edgecolor="white", linewidth=0.5)
    nodes = nx.draw_networkx_nodes(
        graph,
        pos=positions,
        nodelist=STATES,
        node_color=np.asarray(U[:, v]),
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax,
        node_size=250,
        ax=ax,
    )
    nx.draw_networkx_edges(graph, pos=positions, edge_color="gray", ax=ax)
    ax.set(title=f"mode {v}, $\\mu_{{{v}}}$ = {mu[v]:.2f}", xticks=[], yticks=[])
fig.colorbar(nodes, ax=axes, label="$u_v(r)$", shrink=0.6)
fig.suptitle("Graph Laplacian eigenvectors", fontsize=18, fontweight="bold");

## Building Blocks

We now write the construction as small functions. `numpyro.contrib.hsgp` already provides the sine basis and its eigenvalues (`eigenfunctions`, `sqrt_eigenvalues`), the Matérn spectral density evaluated at the eigenvalues (`diag_spectral_density_matern`), and the linear model $\Phi (\sqrt{W} \circ B)$ with its coefficients (`linear_approximation`). We reuse all of them. What we add is the graph side: the map from $\mu_v$ to $(\alpha_v, \ell_v)$, the trace normalization, the node standardization, and the rotation back to the regions.

### Time Side

`time_box` builds the centered grid and the half-width $L = c\,(D - 1) / 2$. `time_basis` returns the sine basis $\Phi$ and the eigenvalues $\lambda_j$ from `numpyro.contrib.hsgp`. Both depend only on the duration and on $m$, not on any hyperparameter.

In [ ]:
def time_box(
    duration: int, c: float = C_BOX
) -> tuple[Float[Array, " duration"], float]:
    "Centered time grid and the box half-width c * (duration - 1) / 2."
    t = jnp.arange(duration, dtype=jnp.float32)
    half = (duration - 1) / 2.0
    return t - half, float(c * half)


def time_basis(
    t_c: Float[Array, " duration"], ell_box: float, m: int
) -> tuple[Float[Array, "duration m"], Float[Array, " m"]]:
    "HSGP sine basis Phi (duration, m) and eigenvalues lambda (m,) on the box."
    phi = eigenfunctions(t_c, ell=ell_box, m=m)
    lam = sqrt_eigenvalues(ell_box, m, dim=1)[0] ** 2
    return phi, lam

### Spectral Weights

The weights are built in three steps.

1. `mode_hyperparameters` maps the graph eigenvalues to the per-mode amplitude and lengthscale $(\alpha_v, \ell_v)$ from the table above. This is the only place where the three priors differ.
2. `diag_spectral_density_matern` from `numpyro.contrib.hsgp` evaluates the one-dimensional Matérn density at $\sqrt{\lambda_j}$ for one $(\alpha, \ell)$ pair; we `vmap` it over the graph modes to get the $(m, R)$ matrix.
3. `trace_normalize` scales the weights so that the average prior variance over the box and the regions is $\sigma_f^2$.

In [ ]:
Mode = Literal["kron_sum", "separable", "independent"]


def mode_hyperparameters(
    mu: Float[Array, " R"],
    *,
    mode: Mode,
    nu: float,
    length: Float[Array, ""],
    rho: Float[Array, ""],
) -> tuple[Float[Array, " R"], Float[Array, " R"]]:
    "Amplitude alpha_v and temporal lengthscale ell_v of every graph mode."
    g = 1.0 + rho * mu
    if mode == "kron_sum":
        return g ** (-nu), length / jnp.sqrt(g)
    if mode == "separable":
        return g ** (-(nu + 0.5)), jnp.full_like(mu, length)
    if mode == "independent":
        return jnp.ones_like(mu), jnp.full_like(mu, length)
    raise ValueError(f"unknown mode {mode!r}")


def trace_normalize(
    w: Float[Array, "m R"], sigma_f: Float[Array, ""], ell_box: float
) -> Float[Array, "m R"]:
    "Scale w so that mean_{t, r} Var f(t, r) = sum w / (2 ell_box R) equals sigma_f**2."
    n_regions = w.shape[1]
    return w * (sigma_f**2 * 2.0 * ell_box * n_regions / jnp.sum(w))


def spectral_weights(
    mu: Float[Array, " R"],
    *,
    mode: Mode,
    nu: float,
    length: Float[Array, ""],
    rho: Float[Array, ""],
    sigma_f: Float[Array, ""],
    ell_box: float,
    m: int,
) -> Float[Array, "m R"]:
    "Trace-normalized spectral weights W[j, v] for one of the three priors."
    alpha_v, length_v = mode_hyperparameters(
        mu, mode=mode, nu=nu, length=length, rho=rho
    )

    def density(alpha, length):
        return diag_spectral_density_matern(
            nu=nu, alpha=alpha, length=length, ell=ell_box, m=m, dim=1
        )

    w = jax.vmap(density)(alpha_v, length_v).T  # (m, R)
    return trace_normalize(w, sigma_f, ell_box)

### Effective Lengthscales and Node Variances

`effective_lengthscales` returns $\ell_v = \ell / \sqrt{1 + \rho \mu_v}$, the temporal lengthscale of each graph mode under the Kronecker sum. We use it for the basis size check. `node_variances` returns the prior variance $v_r$ of $f(\cdot, r)$ at each region, which we need for the node standardization.

In [ ]:
def effective_lengthscales(
    mu: Float[Array, " R"], length: Float[Array, ""], rho: Float[Array, ""]
) -> Float[Array, " R"]:
    "Temporal lengthscale of each graph mode: ell / sqrt(1 + rho mu_v)."
    return length / jnp.sqrt(1.0 + rho * mu)


def node_variances(
    weights: Float[Array, "m R"], U: Float[Array, "R R"], ell_box: float
) -> Float[Array, " R"]:
    "Prior variance v_r = sum_{j, v} W[j, v] U[r, v]**2 / (2 ell_box) at each region."
    return jnp.einsum("jv,rv->r", weights, U**2) / (2.0 * ell_box)

### The GP Block

`kron_hsgp` is the whole latent panel. The first matrix multiplication, $\Phi (\sqrt{W} \circ B)$ with $B \sim \text{Normal}(0, 1)$, is `linear_approximation` from `numpyro.contrib.hsgp`; we call it inside a `graph_mode` plate so that the coefficients get the shape $(m, R)$ (one temporal HSGP per graph mode, site `beta`). The second multiplication rotates back to the regions with the scaled eigenvector matrix $D U$, so the node standardization costs nothing extra. For the independent prior the rotation is skipped ($U = I$).

In [ ]:
def kron_hsgp(
    name: str,
    phi: Float[Array, "T m"],
    U: Float[Array, "R R"],
    weights: Float[Array, "m R"],
    *,
    mode: Mode,
    scale: Float[Array, " R"] | None = None,
) -> Float[Array, "T R"]:
    "Latent panel f = Phi (sqrt(W) * B) (D U)^T, B ~ Normal(0, 1), D = diag(scale)."
    m, R = weights.shape
    with numpyro.plate("graph_mode", R, dim=-1):
        f = linear_approximation(phi, jnp.sqrt(weights), m)  # (T, R), "beta" is (m, R)
    if mode != "independent":
        U_scaled = U if scale is None else U * scale[:, None]  # node standardization
        f = f @ U_scaled.T  # back to regions
    return numpyro.deterministic(f"{name}_f", f)

### The Approximate Kernel

The `approximate_kernel` materializes the full covariance

$$k((t, r), (t', r')) = \sum_{j, v} W_{jv}\, \phi_j(t) \phi_j(t')\, u_v(r) u_v(r')$$

as a $(T, R, T, R)$ array. We never need it for inference; we use it below to look at the prior correlations.

In [ ]:
def approximate_kernel(
    phi: Float[Array, "T m"], U: Float[Array, "R R"], weights: Float[Array, "m R"]
) -> Float[Array, "T R T R"]:
    "Covariance sum_{j, v} W[j, v] phi_j(t) phi_j(t') U[r, v] U[r', v] as (T, R, T, R)."
    return jnp.einsum("tj,sj,rv,qv,jv->trsq", phi, phi, U, U, weights)

## What the Prior Says

Before we look at any data we fix the hyperparameters and inspect the three priors. The forecasting models below use the priors 

\begin{align*}
\sigma_f & \sim \text{HalfNormal}(1), \\
\ell & \sim \text{LogNormal}(\log 20, 0.7), \\
\rho &\sim \text{LogNormal}(0, 1.5).
\end{align*}

We plot them with [PreliZ](https://preliz.readthedocs.io/).

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 5), layout="constrained")
pz.HalfNormal(sigma=1.0).plot_pdf(ax=axes[0], legend=None)
pz.LogNormal(mu=np.log(20.0), sigma=0.7).plot_pdf(ax=axes[1], legend=None)
pz.LogNormal(mu=0.0, sigma=1.5).plot_pdf(ax=axes[2], legend=None)
axes[0].set(title="$\\sigma_f \\sim$ HalfNormal(1)")
axes[1].set(title="$\\ell \\sim$ LogNormal(log 20, 0.7)", xlim=(0, 120))
axes[2].set(title="$\\rho \\sim$ LogNormal(0, 1.5)", xlim=(0, 30))
fig.suptitle("Priors on the GP hyperparameters", fontsize=18, fontweight="bold");

For the prior checks we fix the hyperparameters at values inside the bulk of these priors: $\sigma_f = 1$, $\ell = 20$ weeks, $\rho = 2$. We use the full duration $D = T + H$ with $T = 156$ weeks of history and a horizon of $H = 13$ weeks, a Matérn-$3/2$ kernel ($\nu = 3/2$), and $m = 48$ basis functions.

In [ ]:
prior_hyper = {
    "sigma_f": jnp.asarray(1.0),
    "length": jnp.asarray(20.0),
    "rho": jnp.asarray(2.0),
}

MODES: list[Mode] = ["independent", "separable", "kron_sum"]

weights_fixed = {
    mode: spectral_weights(
        mu, mode=mode, nu=NU, ell_box=ell_box, m=M_BASIS, **prior_hyper
    )
    for mode in MODES
}

### Spectral Weights

The weights $W_{jv}$ are the only thing that changes between the three models. We first look at them as heatmaps over $(j, v)$. For the independent model they do not depend on the graph mode. For the separable model the graph enters as a per-mode amplitude that is the same for every temporal frequency $j$. For the Kronecker sum, rough graph modes lose their high temporal frequencies first: the columns get narrower as $\mu_v$ grows.

In [ ]:
fig, axes = plt.subplots(
    nrows=1, ncols=3, figsize=(16, 5), sharey=True, layout="constrained"
)
log_w = {mode: np.log10(np.asarray(w)) for mode, w in weights_fixed.items()}
vmin = min(w.min() for w in log_w.values())
vmax = max(w.max() for w in log_w.values())
for ax, mode in zip(axes, MODES, strict=True):
    im = ax.imshow(
        log_w[mode], aspect="auto", origin="lower", cmap="viridis", vmin=vmin, vmax=vmax
    )
    ax.set(title=mode, xlabel="graph mode $v$", ylabel="time frequency $j$")
fig.colorbar(im, ax=axes, label="$\\log_{10} W_{jv}$", shrink=0.8)
fig.suptitle("Spectral weights of the three priors", fontsize=18, fontweight="bold");

The same information as curves makes the interpretation visible: for each graph mode $v$, the weights as a function of $j$ are the spectral weights of a temporal Matérn process. In the independent prior all curves coincide. In the separable prior all curves have the same shape and only their level changes. In the Kronecker sum the curves of rough modes are lower *and* flatter, which is a shorter lengthscale.

In [ ]:
modes_to_show = [0, 1, 5, 15]
j_grid = np.arange(1, M_BASIS + 1)

fig, axes = plt.subplots(
    nrows=1, ncols=3, figsize=(16, 5), sharey=True, layout="constrained"
)
for ax, mode in zip(axes, MODES, strict=True):
    for i, v in enumerate(modes_to_show):
        ax.plot(
            j_grid,
            np.asarray(weights_fixed[mode][:, v]),
            color=f"C{i}",
            label=f"$v$ = {v}, $\\mu_v$ = {mu[v]:.2f}",
        )
    ax.set(title=mode, xlabel="time frequency $j$", yscale="log")
axes[0].set(ylabel="$W_{jv}$")
axes[2].legend()
fig.suptitle("Spectral weights per graph mode", fontsize=18, fontweight="bold");

### Effective Lengthscales

Under the Kronecker sum each graph mode is a temporal Matérn process with its own lengthscale $\ell_v = \ell / \sqrt{1 + \rho \mu_v}$. With $\ell = 20$ and $\rho = 2$ the roughest mode on this graph has a lengthscale of about four weeks. The basis size rule must use this value.

In [ ]:
ell_modes = effective_lengthscales(mu, prior_hyper["length"], prior_hyper["rho"])
m_rule = 1.75 * C_BOX * (DURATION / 2) / float(ell_modes.min())

fig, ax = plt.subplots()
ax.bar(np.arange(R), np.asarray(ell_modes), color="C0")
ax.axhline(
    y=float(prior_hyper["length"]),
    color="C1",
    linestyle="--",
    label="$\\ell$ (national mode)",
)
ax.set(xlabel="graph mode $v$", ylabel="$\\ell_v$ (weeks)", xticks=np.arange(R))
ax.legend()
fig.suptitle(
    "Effective temporal lengthscale per graph mode", fontsize=18, fontweight="bold"
);

In [ ]:
print(f"shortest effective lengthscale: {float(ell_modes.min()):.1f} weeks")
print(f"basis size rule: m >= {m_rule:.0f} (we use m = {M_BASIS})")

The rule asks for slightly more basis functions than we use. We keep $m = 48$ for speed and come back to this point after the fit, where we check the rule against the posterior and refit with a larger $m$.

### Non-Separability

What does non-separable mean in practice? For a separable kernel $k((t, r), (t', r')) = k_t(t - t')\, k_G(r, r')$, the correlation between two regions at time lag $k$ factorizes,

$$
\text{Corr}\big(f(t, r), f(t + k, r')\big) = \frac{k_t(k)}{k_t(0)} \cdot \frac{k_G(r, r')}{\sqrt{k_G(r, r)\, k_G(r', r')}},
$$

so the *ratio* of the correlation between adjacent regions to the correlation between non-adjacent regions does not depend on the lag. For the Kronecker sum, rough spatial modes decay faster in time, so at long lags only the smooth national mode survives and all regions become equally correlated: the ratio moves toward one. We check this directly on the approximate kernel. We standardize the nodes first so that all regions have the same variance, and we define the two sets of region pairs.

In [ ]:
def standardized_U(
    weights: Float[Array, "m R"],
    U: Float[Array, "R R"],
    ell_box: float,
    sigma_f: Float[Array, ""],
) -> Float[Array, "R R"]:
    "Eigenvector matrix D U with the node standardization folded in."
    scale = sigma_f / jnp.sqrt(node_variances(weights, U, ell_box))
    return U * scale[:, None]


adjacent = adjacency.astype(bool)
non_adjacent = ~adjacent & ~np.eye(R, dtype=bool)

Next we compute the full kernel, turn it into a correlation, and average the correlation at lag $k$ over adjacent and over non-adjacent pairs, with the reference time $t$ in the middle of the box.

In [ ]:
def correlation_by_lag(mode: Mode, lags: np.ndarray, t0: int) -> dict[str, np.ndarray]:
    "Mean prior correlation between regions at each lag, by adjacency."
    U_mode = standardized_U(weights_fixed[mode], U, ell_box, prior_hyper["sigma_f"])
    kernel = np.asarray(approximate_kernel(phi_full, U_mode, weights_fixed[mode]))
    sd = np.sqrt(np.einsum("trtr->tr", kernel))
    corr = kernel / (sd[:, :, None, None] * sd[None, None, :, :])
    return {
        "adjacent": np.array([corr[t0, :, t0 + k, :][adjacent].mean() for k in lags]),
        "non-adjacent": np.array(
            [corr[t0, :, t0 + k, :][non_adjacent].mean() for k in lags]
        ),
    }


lags = np.arange(0, 27)
corr_by_lag = {
    mode: correlation_by_lag(mode, lags, t0=DURATION // 2)
    for mode in ["separable", "kron_sum"]
}

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), layout="constrained")
for i, mode in enumerate(["separable", "kron_sum"]):
    for j, pair in enumerate(["adjacent", "non-adjacent"]):
        axes[0].plot(
            lags,
            corr_by_lag[mode][pair],
            color=f"C{i}",
            linestyle=["-", "--"][j],
            label=f"{mode}, {pair}",
        )
    ratio = corr_by_lag[mode]["non-adjacent"] / corr_by_lag[mode]["adjacent"]
    axes[1].plot(lags, ratio, color=f"C{i}", label=mode)
axes[0].set(
    xlabel="time lag (weeks)",
    ylabel="prior correlation",
    title="Cross-region correlation by lag",
)
axes[0].legend()
axes[1].set(
    xlabel="time lag (weeks)",
    ylabel="non-adjacent / adjacent",
    title="Ratio non-adjacent to adjacent",
)
axes[1].legend()
fig.suptitle("Separable versus Kronecker-sum prior", fontsize=18, fontweight="bold");

For the separable prior the ratio is flat: the spatial pattern of the correlations is the same at every lag. For the Kronecker sum the ratio increases with the lag: far apart in time, all regions look alike because only the national mode has memory that long. This is the behavior we want for a forecast: as the horizon grows, the regional forecasts revert toward the national one.

### Node Standardization

The graph Matérn prior does not have the same variance at every node. We compute $v_r$ for the Kronecker-sum prior before and after the standardization.

In [ ]:
v_raw = node_variances(weights_fixed["kron_sum"], U, ell_box)
U_std = standardized_U(weights_fixed["kron_sum"], U, ell_box, prior_hyper["sigma_f"])
v_std = node_variances(weights_fixed["kron_sum"], U_std, ell_box)

fig, ax = plt.subplots()
x = np.arange(R)
ax.bar(x - 0.2, np.asarray(v_raw), width=0.4, color="C0", label="raw")
ax.bar(x + 0.2, np.asarray(v_std), width=0.4, color="C1", label="standardized")
ax.axhline(
    y=float(prior_hyper["sigma_f"]) ** 2,
    color="black",
    linestyle="--",
    label="$\\sigma_f^2$",
)
ax.set(xticks=x, xticklabels=STATES, ylabel="prior variance $v_r$", xlabel="region")
ax.legend()
fig.suptitle("Prior variance per region (kron_sum)", fontsize=18, fontweight="bold");

In [ ]:
v_min, v_max = float(v_raw.min()), float(v_raw.max())
print(f"raw variance range: [{v_min:.2f}, {v_max:.2f}] (ratio {v_max / v_min:.1f})")
print(f"standardized range: [{float(v_std.min()):.2f}, {float(v_std.max()):.2f}]")

The raw prior gives the low-degree states (Bremen, Saarland, Berlin) a much wider prior band than the hubs. After the standardization every region has variance $\sigma_f^2$ and the graph only shapes the correlations.

### Prior Draws

Finally we draw one latent panel from each prior. `latent_prior` is a small NumPyro model with the three hyperparameters as sample sites and the GP block from above. We fix the hyperparameters with the `substitute` handler, which replaces the value of a sample site without changing the random stream. Because the three models draw the coefficients $\beta_{jv}$ from the same site with the same key, the three panels use the *same* coefficients and differ only in the weights, so they are directly comparable.

In [ ]:
def latent_prior(mode: Mode, m: int = M_BASIS, nu: float = NU):
    "Prior on the latent panel f with the three hyperparameters as sample sites."
    phi_m, _ = time_basis(t_centered, ell_box, m)

    def model():
        sigma_f = numpyro.sample("sigma_f", dist.HalfNormal(1.0))
        length = numpyro.sample("length", dist.LogNormal(jnp.log(20.0), 0.7))
        rho = numpyro.sample("rho", dist.LogNormal(0.0, 1.5))
        w = spectral_weights(
            mu,
            mode=mode,
            nu=nu,
            length=length,
            rho=rho,
            sigma_f=sigma_f,
            ell_box=ell_box,
            m=m,
        )
        scale = (
            None
            if mode == "independent"
            else sigma_f / jnp.sqrt(node_variances(w, U, ell_box))
        )
        kron_hsgp("gp", phi_m, U, w, mode=mode, scale=scale)

    return model

In [ ]:
rng_key, rng_subkey = random.split(rng_key)
prior_draws = {
    mode: Predictive(substitute(latent_prior(mode), data=prior_hyper), num_samples=1)(
        rng_subkey
    )["gp_f"][0]
    for mode in MODES
}

In [ ]:
fig, axes = plt.subplots(
    nrows=3, ncols=1, figsize=(12, 12), sharex=True, sharey=True, layout="constrained"
)
for ax, mode in zip(axes, MODES, strict=True):
    f_draw = np.asarray(prior_draws[mode])
    for r in range(R):
        ax.plot(f_draw[:, r], color="C0", alpha=0.4)
    ax.plot(f_draw.mean(axis=1), color="C1", linewidth=3, label="regional mean")
    ax.axvline(x=T_OBS, color="black", linestyle="--")
    ax.set(title=mode, ylabel="$f(t, r)$")
axes[0].legend(loc="upper left")
axes[-1].set(xlabel="week")
fig.suptitle(
    "One prior draw per model (same coefficients)", fontsize=18, fontweight="bold"
);

The independent draw has no common structure across regions, and the regional mean is damped. The separable draw pools the regions, and every regional deviation is as smooth as the trend, because all graph modes share the lengthscale $\ell$. In the Kronecker-sum draw the regions share the same trend, but the deviations have a shorter memory: they move faster and keep reverting to the common shape. This is what a diffusion on the network looks like.

## Data Generating Process

We simulate the panel with a generative model that is deliberately different from the forecasting model, so that the comparison below is not a parameter-recovery exercise. The panel has four components.

**National trend.** One draw of a Matérn-$5/2$ Gaussian process in time with lengthscale $\ell_g$ and scale $\sigma_g$ (we sample it as an HSGP with a large basis):

$$
g \sim \text{GP}\big(0, k_{5/2}\big), \qquad k_{5/2}(\tau) = \sigma_g^2 \left(1 + \frac{\sqrt{5} |\tau|}{\ell_g} + \frac{5 \tau^2}{3 \ell_g^2}\right) \exp\left(-\frac{\sqrt{5} |\tau|}{\ell_g}\right).
$$

**Regional deviations.** A graph Ornstein-Uhlenbeck process, a diffusion on the network with mean reversion $\theta$ and diffusion rate $a$:

$$
x_{t + 1} = e^{-(\theta I + a L_G)}\, x_t + \varepsilon_t, \qquad \varepsilon_t \sim \text{Normal}(0, s^2 I).
$$

In graph-spectral coordinates $z_t = U^\top x_t$ this is an independent AR(1) process per graph mode with coefficient $\alpha_v = e^{-(\theta + a \mu_v)}$. Its stationary law is Gaussian with an exponential (Matérn-$1/2$) covariance in time:

$$
\text{Cov}(z_{t, v}, z_{t', v}) = \frac{s^2}{1 - \alpha_v^2}\, \alpha_v^{|t - t'|}, \qquad x_t = U z_t .
$$

So the deviations are rougher in time ($\nu = 1/2$) than the model assumes ($\nu = 3/2$), and their variance profile over the graph modes differs from the Matérn one. We set the constant mode to zero because the national trend already covers it.

**Seasonality.** A yearly Fourier pair with fixed weights, $s(t) = w^\top \text{fourier}(t)$, with the same features as the model.

**Observations.** Regional intercepts $a_r \sim \text{Normal}(0, 0.5)$ and Student-t noise:

$$
y_{t, r} \sim \text{StudentT}\big(\nu_y,\; a_r + g_t + s(t) + x_{t, r},\; \sigma_y\big).
$$

We write the simulation as a NumPyro model with the hyperparameters as sample sites, fix them with the `do` handler, and take one draw. As in the [NumPyro HSGP example](https://num.pyro.ai/en/stable/examples/hsgp.html), each component is its own function with its own sample sites, and the model only composes them.

The national trend is a Matérn-$5/2$ HSGP with a large basis, sampled with `hsgp_matern` from `numpyro.contrib.hsgp` (its coefficient site is called `beta`). The amplitude argument of `hsgp_matern` is the variance $\sigma_g^2$.

In [ ]:
M_TREND = 100


def national_trend(t_c: Float[Array, " T"], ell_box: float) -> Float[Array, " T"]:
    "One Matérn-5/2 HSGP draw shared by all regions."
    trend_length = numpyro.sample("trend_length", dist.LogNormal(jnp.log(40.0), 0.5))
    trend_scale = numpyro.sample("trend_scale", dist.HalfNormal(1.0))
    trend = hsgp_matern(
        x=t_c, nu=2.5, alpha=trend_scale**2, length=trend_length, ell=ell_box, m=M_TREND
    )
    return numpyro.deterministic("trend", trend)

The regional deviations follow the stationary law of the graph OU process: one multivariate normal per non-constant graph mode with the covariance derived above, rotated back to the regions with $U$.

In [ ]:
def graph_ou_covariance(
    dt: Float[Array, "T T"],
    mu: Float[Array, " V"],
    theta: Array,
    diffusion: Array,
    noise: Array,
) -> Float[Array, "V T T"]:
    "Stationary covariance of the graph OU process, one (T, T) matrix per graph mode."
    alpha = jnp.exp(-(theta + diffusion * mu))  # AR(1) coefficient per mode
    variance = noise**2 / (1 - alpha**2)
    return variance[:, None, None] * alpha[:, None, None] ** dt[None]


def regional_deviations(
    t: Float[Array, " T"], mu: Float[Array, " R"], U: Float[Array, "R R"]
) -> Float[Array, "T R"]:
    "Graph OU deviations: independent AR(1) per graph mode, rotated to the regions."
    theta = numpyro.sample("theta", dist.LogNormal(jnp.log(0.1), 0.5))
    diffusion = numpyro.sample("diffusion", dist.LogNormal(jnp.log(0.05), 0.5))
    dev_noise = numpyro.sample("dev_noise", dist.HalfNormal(0.5))
    dt = jnp.abs(t[:, None] - t[None, :])
    k_dev = graph_ou_covariance(dt, mu[1:], theta, diffusion, dev_noise)
    jitter = 1e-4 * jnp.eye(t.shape[0])
    z = numpyro.sample(
        "z", dist.MultivariateNormal(jnp.zeros(t.shape[0]), k_dev + jitter)
    )
    return numpyro.deterministic("dev", z.T @ U[:, 1:].T)  # (T, R)

Seasonality is a yearly Fourier pair, and the observations are Student-t around the latent panel.

In [ ]:
def seasonality(duration: int, period: float) -> Float[Array, " T"]:
    "Yearly seasonality from two Fourier terms with sampled weights."
    w_seas = numpyro.sample("w_seas", dist.Normal(0.0, 0.5).expand([4]).to_event(1))
    return fourier_features(duration, period=period, num_terms=2) @ w_seas


def observe(latent: Float[Array, "T R"]) -> None:
    "Student-t observations around the latent panel."
    obs_scale = numpyro.sample("obs_scale", dist.HalfNormal(0.5))
    obs_df = numpyro.sample("obs_df", dist.Gamma(4.0, 0.5))
    numpyro.sample("y", dist.StudentT(obs_df, latent, obs_scale).to_event(2))

The panel model composes the four components.

In [ ]:
def panel_dgp(
    mu: Float[Array, " R"],
    U: Float[Array, "R R"],
    duration: int,
    period: float = PERIOD,
) -> None:
    "Weekly panel: trend + graph OU deviations + seasonality + Student-t noise."
    t_c, ell_box = time_box(duration)
    trend = national_trend(t_c, ell_box)
    dev = regional_deviations(t_c, mu, U)
    seasonal = seasonality(duration, period)
    intercept = numpyro.sample(
        "intercept", dist.Normal(0.0, 0.5).expand([mu.shape[0]]).to_event(1)
    )
    latent = numpyro.deterministic(
        "latent", intercept + trend[:, None] + seasonal[:, None] + dev
    )
    observe(latent)

These are the true values. The seasonal weights give $0.6 \sin(2 \pi t / 52) + 0.25 \cos(4 \pi t / 52)$.

In [ ]:
true_values = {
    "trend_length": 40.0,
    "trend_scale": 1.0,
    "theta": 0.08,
    "diffusion": 0.05,
    "dev_noise": 0.35,
    "w_seas": jnp.array([0.6, 0.0, 0.0, 0.25]),
    "obs_scale": 0.25,
    "obs_df": 6.0,
}

pl.DataFrame(
    {
        "parameter": list(true_values),
        "value": [str(np.asarray(v)) for v in true_values.values()],
    }
)

Before we draw, we look at what the graph OU process does to each graph mode: the AR(1) coefficient $\alpha_v$ (how long a deviation in that mode persists) and the stationary standard deviation. Smooth modes persist longer and carry more variance, which is the qualitative structure the Kronecker sum encodes, but with a different functional form.

In [ ]:
alpha_modes = np.exp(
    -(true_values["theta"] + true_values["diffusion"] * np.asarray(mu[1:]))
)
sd_modes = true_values["dev_noise"] / np.sqrt(1 - alpha_modes**2)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), layout="constrained")
axes[0].bar(np.arange(1, R), alpha_modes, color="C0")
axes[0].set(
    xlabel="graph mode $v$",
    ylabel="$\\alpha_v$",
    title="AR(1) coefficient per mode",
    ylim=(0, 1),
    xticks=np.arange(1, R),
)
axes[1].bar(np.arange(1, R), sd_modes, color="C1")
axes[1].set(
    xlabel="graph mode $v$",
    ylabel="stationary sd",
    title="Stationary standard deviation per mode",
    ylim=(0, 1),
    xticks=np.arange(1, R),
)
fig.suptitle(
    "Graph OU process in spectral coordinates", fontsize=18, fontweight="bold"
);

In [ ]:
rng_key, rng_subkey = random.split(rng_key)
simulation = Predictive(do(panel_dgp, data=true_values), num_samples=1)(
    rng_subkey, mu, U, DURATION
)

y = simulation["y"][0]  # (DURATION, R)
latent_true = simulation["latent"][0]
trend_true = simulation["trend"][0]
dev_true = simulation["dev"][0]

print(f"panel shape (weeks, regions) = {y.shape}")

We look at the components of the simulation.

In [ ]:
fig, axes = plt.subplots(
    nrows=3, ncols=1, figsize=(12, 12), sharex=True, layout="constrained"
)
axes[0].plot(np.asarray(trend_true), color="C0")
axes[0].set(title="National trend (Matérn-5/2 draw)", ylabel="trend")
for r in range(R):
    axes[1].plot(np.asarray(dev_true[:, r]), color="C0", alpha=0.4)
axes[1].set(
    title="Regional deviations (graph OU), one line per region", ylabel="deviation"
)
for r in range(R):
    axes[2].plot(np.asarray(y[:, r]), color="C0", alpha=0.4)
axes[2].axvline(x=T_OBS, color="black", linestyle="--", label="train-test split")
axes[2].set(title="Observed panel, one line per region", ylabel="y", xlabel="week")
axes[2].legend(loc="upper left")
fig.suptitle("Simulated panel", fontsize=18, fontweight="bold");

## Train-Test Split

We keep the first $T = 156$ weeks for training and the last $H = 13$ weeks as the test set. The covariates are the Fourier features of the yearly seasonality over the full duration. Following the `numpyro-forecast` convention, time is at axis $-2$ and the regions at axis $-1$.

In [ ]:
covariates = fourier_features(DURATION, period=PERIOD, num_terms=2)  # (DURATION, 4)
covariates_train = covariates[:T_OBS]

y_train = y[:T_OBS]
y_test = y[T_OBS:]

print(f"train {y_train.shape}, test {y_test.shape}, covariates {covariates.shape}")

## The Forecasting Models

One model factory produces the three models. The model follows the `numpyro-forecast` convention: it takes `(covariates, data)`, derives the train and forecast split with `Horizon.from_data`, builds the prediction over the full covariate duration, and hands it to `predict`, which observes the in-sample prefix and samples the forecast suffix.

Two details matter.

- **The time box is fixed once.** The basis must be the same at training time and at forecast time. If we built the box from the length of the current call, the training call (156 weeks) and the forecast call (169 weeks) would use different boxes, and the posterior coefficients would be read against the wrong basis. So the factory takes the full `duration`, builds the basis once, and each call slices the prefix it needs.
- **The node standardization is folded into $U$.** We multiply the rows of $U$ by the scales $s_r$ before the second matrix multiplication (this is the $D U$ of the $D K D$ kernel), so the GP block stays two matrix multiplications.

The priors are the ones plotted above for $(\sigma_f, \ell, \rho)$ ($\rho$ is absent in the independent model), plus regional intercepts $a_r \sim \text{Normal}(0, 1)$, seasonal weights $\text{Normal}(0, 0.5)$, $\sigma \sim \text{HalfNormal}(0.5)$, and $\nu_{\varepsilon} \sim \text{Gamma}(4, 0.5)$.

In [ ]:
def make_model(
    mu: Float[Array, " R"],
    U: Float[Array, "R R"],
    *,
    mode: Mode,
    duration: int,
    m: int = M_BASIS,
    c: float = C_BOX,
    nu: float = NU,
):
    "Forecasting model factory. The time box covers `duration` steps and is fixed."
    R = mu.shape[0]
    t_c, ell_box = time_box(duration, c=c)
    phi_full, _ = time_basis(t_c, ell_box, m)

    def model(
        covariates: Float[Array, "duration K"], data: Float[Array, "T R"] | None = None
    ) -> None:
        h = Horizon.from_data(covariates, data)
        phi = phi_full[: h.duration]  # prefix of the fixed basis
        # gp hyperparameters
        sigma_f = numpyro.sample("sigma_f", dist.HalfNormal(1.0))
        length = numpyro.sample("length", dist.LogNormal(jnp.log(20.0), 0.7))
        rho = (
            numpyro.sample("rho", dist.LogNormal(0.0, 1.5))
            if mode != "independent"
            else jnp.zeros(())
        )
        # latent panel
        w = spectral_weights(
            mu,
            mode=mode,
            nu=nu,
            length=length,
            rho=rho,
            sigma_f=sigma_f,
            ell_box=ell_box,
            m=m,
        )
        scale = (
            None
            if mode == "independent"
            else sigma_f / jnp.sqrt(node_variances(w, U, ell_box))
        )
        f = kron_hsgp("gp", phi, U, w, mode=mode, scale=scale)  # (duration, R)
        # intercepts and seasonality
        intercept = numpyro.sample(
            "intercept", dist.Normal(0.0, 1.0).expand([R]).to_event(1)
        )
        w_seas = numpyro.sample(
            "w_seas", dist.Normal(0.0, 0.5).expand([covariates.shape[-1]]).to_event(1)
        )
        prediction = intercept + f + (covariates @ w_seas)[:, None]
        # likelihood
        sigma = numpyro.sample("sigma", dist.HalfNormal(0.5))
        df = numpyro.sample("df", dist.Gamma(4.0, 0.5))
        predict(h, dist.StudentT(df, 0.0, sigma), prediction)

    return model


models = {mode: make_model(mu, U, mode=mode, duration=DURATION) for mode in MODES}

### Prior Predictive Check

We draw from the prior predictive of the Kronecker-sum model over the training window to check that the implied scale of the observations is reasonable.

In [ ]:
rng_key, rng_subkey = random.split(rng_key)
prior_predictive = Predictive(models["kron_sum"], num_samples=500)(
    rng_subkey, covariates_train
)

prior_obs = np.asarray(prior_predictive["obs"])  # (draws, T_OBS, R)

fig, ax = plt.subplots()
for i in range(20):
    ax.plot(prior_obs[i, :, state_index["BY"]], color="C0", alpha=0.3)
ax.plot(np.asarray(y_train[:, state_index["BY"]]), color="black", label="observed (BY)")
ax.legend(loc="upper left")
ax.set(xlabel="week", ylabel="y")
fig.suptitle("Prior predictive draws for Bavaria", fontsize=18, fontweight="bold");

## Model Fit and Diagnostics

We fit the three models with NUTS (4 chains, 1000 warmup and 1000 posterior draws each). Everything in the model is Gaussian conditional on a handful of hyperparameters, so the posterior is well behaved.

In [ ]:
%%time


def fit_nuts(
    rng_key,
    model,
    covariates,
    data,
    *,
    num_warmup: int = 1_000,
    num_samples: int = 1_000,
    num_chains: int = 4,
) -> MCMC:
    mcmc = MCMC(
        NUTS(model, init_strategy=init_to_median, target_accept_prob=0.9),
        num_warmup=num_warmup,
        num_samples=num_samples,
        num_chains=num_chains,
        progress_bar=False,
    )
    mcmc.run(rng_key, covariates, data)
    return mcmc


def to_idata(mcmc: MCMC) -> az.InferenceData:
    "Convert the MCMC run to arviz, with numpy-backed arrays."
    return az.from_numpyro(mcmc).map_over_datasets(lambda ds: ds.as_numpy())


mcmcs: dict[str, MCMC] = {}
idatas: dict[str, az.InferenceData] = {}
for mode in MODES:
    rng_key, rng_subkey = random.split(rng_key)
    t_start = time.time()
    mcmcs[mode] = fit_nuts(rng_subkey, models[mode], covariates_train, y_train)
    idatas[mode] = to_idata(mcmcs[mode])
    print(f"{mode:12s} fitted in {time.time() - t_start:.0f} s")

In [ ]:
for mode in MODES:
    print(f"--- {mode}")
    az.diagnose(idatas[mode])

The chains mix well: no divergences, and satisfactory effective sample sizes and $\hat{R}$ for all parameters in all models. We compare the posteriors of the hyperparameters across the three models. The shared parameters live on very different scales, so we overlay their densities one variable at a time, and we use a forest plot for the two parameters that control the pooling.

In [ ]:
hyper_names = ["sigma_f", "length", "rho", "sigma", "df"]

pc = az.plot_dist(
    idatas,
    var_names=["sigma_f", "length", "sigma", "df"],
    figure_kwargs={"figsize": (16, 4)},
)
pc.add_legend("model")
pc.viz["figure"].item().suptitle(
    "Posterior of the shared hyperparameters", fontsize=18, fontweight="bold", y=1.08
);

In [ ]:
pc = az.plot_forest(
    {"separable": idatas["separable"], "kron_sum": idatas["kron_sum"]},
    var_names=["length", "rho"],
    combined=True,
    figure_kwargs={"figsize": (10, 4)},
)
pc.add_legend("model")
pc.viz["figure"].item().suptitle(
    "Lengthscale and pooling strength", fontsize=18, fontweight="bold", y=1.05
);

Note how the three models disagree on the temporal lengthscale: the independent and separable models must use a single $\ell$ for everything, and they choose a short one because the regional deviations are rough. The Kronecker-sum model assigns the short memory to the rough spatial modes through $\rho$ and keeps a longer $\ell$ for the national trend. The summary of the Kronecker-sum model gives the numbers.

In [ ]:
pl.from_pandas(
    az.summary(
        idatas["kron_sum"], var_names=hyper_names, ci_prob=HDI_PROB, ci_kind="hdi"
    ).reset_index(names="parameter")
)

In [ ]:
pc = az.plot_trace_dist(
    idatas["kron_sum"], var_names=hyper_names, figure_kwargs={"figsize": (12, 10)}
)
pc.viz["figure"].item().suptitle(
    "Trace (kron_sum)", fontsize=18, fontweight="bold", y=1.03
);

### Basis Size Check

We now check the basis size rule against the posterior of the Kronecker-sum model. We compute the effective lengthscale of each graph mode at the posterior median.

In [ ]:
posterior_kron = mcmcs["kron_sum"].get_samples()
length_hat = float(jnp.median(posterior_kron["length"]))
rho_hat = float(jnp.median(posterior_kron["rho"]))
ell_post = np.asarray(effective_lengthscales(mu, length_hat, rho_hat))
m_rule_post = 1.75 * C_BOX * (DURATION / 2) / ell_post.min()

print(f"posterior median: length = {length_hat:.1f} weeks, rho = {rho_hat:.2f}")
print(
    f"effective lengthscales: {ell_post[0]:.1f} (national) to {ell_post[-1]:.1f} weeks"
)
print(f"basis size rule: m >= {m_rule_post:.0f} (we used m = {M_BASIS})")

The rule asks for more basis functions than we used, so the roughest modes are under-resolved. We refit the Kronecker-sum model with $m = 96$ and compare the hyperparameters and the forecast scores below. If the posterior of $\rho$ moves down with the larger basis, the truncation was acting as extra pooling.

In [ ]:
M_LARGE = 96
models["kron_sum_m96"] = make_model(
    mu, U, mode="kron_sum", duration=DURATION, m=M_LARGE
)

rng_key, rng_subkey = random.split(rng_key)
t_start = time.time()
mcmcs["kron_sum_m96"] = fit_nuts(
    rng_subkey, models["kron_sum_m96"], covariates_train, y_train
)
idatas["kron_sum_m96"] = to_idata(mcmcs["kron_sum_m96"])
print(f"kron_sum with m = {M_LARGE} fitted in {time.time() - t_start:.0f} s")

pc = az.plot_forest(
    {"m = 48": idatas["kron_sum"], "m = 96": idatas["kron_sum_m96"]},
    var_names=["length", "rho"],
    combined=True,
    figure_kwargs={"figsize": (10, 4)},
)
pc.add_legend("model", title="basis size")
pc.viz["figure"].item().suptitle(
    "Kronecker-sum posterior for two basis sizes",
    fontsize=18,
    fontweight="bold",
    y=1.05,
);

With the larger basis the posterior of $\rho$ moves down a little and $\ell$ gets shorter: part of the pooling in the $m = 48$ fit came from the truncation, which could not represent the fastest regional deviations and pushed them into the shared trend and into the noise. The two posteriors overlap, so the effect is mild here. We keep both fits and score them below.

## Forecasts

The `forecast` function of `numpyro-forecast` runs the model with the full covariates and the training data: the in-sample sites come from the posterior and the forecast suffix is sampled at `obs_future`. We pass only the latent sites of the posterior (not the deterministic `gp_f`, whose in-sample shape would not match the full horizon).

In [ ]:
LATENT_SITES = [
    "sigma_f",
    "length",
    "rho",
    "beta",
    "intercept",
    "w_seas",
    "sigma",
    "df",
]

forecasts: dict[str, Array] = {}
for name, mcmc in mcmcs.items():
    posterior = {k: v for k, v in mcmc.get_samples().items() if k in LATENT_SITES}
    rng_key, rng_subkey = random.split(rng_key)
    forecasts[name] = forecast(
        rng_subkey, models[name], posterior, y_train, covariates
    )  # (draws, H, R)

print({name: tuple(v.shape) for name, v in forecasts.items()})

We plot the forecasts for four regions: a hub (NI), a leaf (HB), and two states in the east and the south (SN, BY). The bands are the central $94\%$ intervals of the forecast draws.

In [ ]:
def plot_forecast_panel(
    forecasts: dict[str, Array], names: list[str], regions: list[str], history: int = 52
) -> None:
    fig, axes = plt.subplots(
        nrows=len(regions),
        ncols=len(names),
        figsize=(16, 3.5 * len(regions)),
        sharex=True,
        sharey="row",
        layout="constrained",
    )
    t_hist = np.arange(T_OBS - history, T_OBS)
    t_fut = np.arange(T_OBS, DURATION)
    for row, region in enumerate(regions):
        r = state_index[region]
        for col, name in enumerate(names):
            ax = axes[row, col]
            ax.plot(
                t_hist, np.asarray(y_train[-history:, r]), color="black", label="train"
            )
            ax.plot(
                t_fut,
                np.asarray(y_test[:, r]),
                color="black",
                linestyle="--",
                marker="o",
                markersize=3,
                label="test",
            )
            draws = np.asarray(forecasts[name][:, :, r])
            lower, upper = np.quantile(
                draws, [(1 - HDI_PROB) / 2, 1 - (1 - HDI_PROB) / 2], axis=0
            )
            ax.fill_between(
                t_fut,
                lower,
                upper,
                color=f"C{col}",
                alpha=0.3,
                label=f"{HDI_PROB:.0%} interval",
            )
            ax.plot(
                t_fut,
                np.median(draws, axis=0),
                color=f"C{col}",
                label="median forecast",
            )
            ax.axvline(x=T_OBS, color="gray", linestyle=":")
            ax.set(title=f"{region}: {name}")
        axes[row, 0].set(ylabel="y")
    axes[0, 0].legend(loc="lower left", fontsize=9)
    for ax in axes[-1]:
        ax.set(xlabel="week")
    fig.suptitle("13-week forecasts", fontsize=18, fontweight="bold")


plot_forecast_panel(forecasts, names=MODES, regions=["NI", "HB", "SN", "BY"]);

## Forecast Evaluation

We score the forecast panel with the CRPS (lower is better), the RMSE of the mean forecast, and the empirical coverage of the central $90\%$ interval. We also report the CRPS on the last four weeks of the horizon, where spatial pooling should matter most.

In [ ]:
def score(samples: Array, truth: Array) -> dict[str, float]:
    return {
        "crps": float(eval_crps(samples, truth)),
        "crps_h10_13": float(eval_crps(samples[:, -4:], truth[-4:])),
        "rmse": float(eval_rmse(samples, truth)),
        "coverage_90": float(eval_coverage(samples, truth, alpha=0.9)),
    }


scores = pl.DataFrame(
    [
        {
            "model": name,
            **score(samples, y_test),
            "length": float(jnp.median(mcmcs[name].get_samples()["length"])),
            "rho": float(jnp.median(mcmcs[name].get_samples()["rho"]))
            if "rho" in mcmcs[name].get_samples()
            else None,
        }
        for name, samples in forecasts.items()
    ]
)
scores

In [ ]:
crps_by_h = {
    name: np.array([float(eval_crps(samples[:, h], y_test[h])) for h in range(H)])
    for name, samples in forecasts.items()
}

fig, ax = plt.subplots()
for i, name in enumerate(forecasts):
    ax.plot(np.arange(1, H + 1), crps_by_h[name], marker="o", color=f"C{i}", label=name)
ax.set(xlabel="horizon (weeks ahead)", ylabel="CRPS", xticks=np.arange(1, H + 1))
ax.legend()
fig.suptitle("CRPS by horizon step", fontsize=18, fontweight="bold");

On this split the Kronecker-sum model is clearly better at every horizon, and the gap grows with the horizon. The forecast panels show why. The national trend turns down in the last year of the training data and stays low over the test period. The independent and separable models use one short lengthscale for everything (about 8 to 11 weeks), so their forecasts revert toward the regional intercepts within a few weeks, up and away from the data. The Kronecker-sum model keeps a long lengthscale for the national mode (about 22 weeks) and extrapolates the level of the trend, while the short memory goes to the rough spatial modes. The larger basis does not change this picture. A single split is one draw of a noisy statistic, and how much the ranking depends on the trend at the split is exactly what a backtest tells us. Before that we look at the spatial structure of the forecasts, which is where the three priors differ by construction.

### Spatial Structure of the Forecasts

The three priors differ in how the forecast of one region is tied to the forecast of its neighbors. We compute, at every horizon step, the correlation across forecast draws between pairs of regions, and we average it over adjacent and over non-adjacent pairs. Close to the data every forecast is pinned down by the observations, so all correlations start near zero. As the horizon grows, the shared components of the uncertainty take over, and the shape of that growth is the signature of each prior.

In [ ]:
def spatial_correlation(samples: Array) -> dict[str, np.ndarray]:
    "Mean correlation across draws for adjacent and non-adjacent regions, per horizon."
    out = {"adjacent": [], "non-adjacent": []}
    for h in range(samples.shape[1]):
        corr = np.corrcoef(np.asarray(samples[:, h, :]).T)
        out["adjacent"].append(corr[adjacent].mean())
        out["non-adjacent"].append(corr[non_adjacent].mean())
    return {k: np.array(v) for k, v in out.items()}


spatial_corr = {name: spatial_correlation(forecasts[name]) for name in MODES}

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5), layout="constrained")
for i, name in enumerate(MODES):
    axes[0].plot(
        np.arange(1, H + 1),
        spatial_corr[name]["adjacent"],
        color=f"C{i}",
        marker="o",
        label=f"{name}, adjacent",
    )
    axes[0].plot(
        np.arange(1, H + 1),
        spatial_corr[name]["non-adjacent"],
        color=f"C{i}",
        marker="o",
        linestyle="--",
        label=f"{name}, non-adjacent",
    )
    axes[1].plot(
        np.arange(1, H + 1),
        spatial_corr[name]["adjacent"] - spatial_corr[name]["non-adjacent"],
        color=f"C{i}",
        marker="o",
        label=name,
    )
axes[0].set(
    xlabel="horizon (weeks ahead)",
    ylabel="correlation of forecast draws",
    title="Adjacent and non-adjacent pairs",
    xticks=np.arange(1, H + 1),
)
axes[0].legend()
axes[1].set(
    xlabel="horizon (weeks ahead)",
    ylabel="adjacent minus non-adjacent",
    title="Gap between the two",
    xticks=np.arange(1, H + 1),
)
axes[1].legend()
fig.suptitle("Spatial correlation of the forecasts", fontsize=18, fontweight="bold");

For the independent model the only shared components are the seasonal weights, so both curves stay near zero. For the separable model the correlations saturate after about eight weeks, with adjacent pairs at about $0.4$ and non-adjacent pairs at about $0.1$: the spatial pattern of the forecast uncertainty is fixed. For the Kronecker sum the correlations keep growing over the whole horizon, for adjacent *and* for non-adjacent pairs. The smooth spatial modes and the national mode have the longest memory, so the further we forecast, the more the regional forecasts move as one block around the national forecast.

## Rolling-Origin Backtest

We use `backtest` from `numpyro-forecast` with six expanding windows, a 13-week test window, and a stride of four weeks. The `forecast_fn` closure fits the model with NUTS on the training window and forecasts the test window. Since the box of every model covers the full duration, the same factory serves every window: each call only sees a shorter prefix of the basis.

In [ ]:
def forecast_fn(
    rng_key,
    model,
    train_data,
    train_covariates,
    full_covariates,
    num_samples,
    *,
    batch_size=None,
):
    k_fit, k_fc = random.split(rng_key)
    mcmc = fit_nuts(
        k_fit,
        model,
        train_covariates,
        train_data,
        num_warmup=500,
        num_samples=num_samples // 4,
    )  # 4 chains
    posterior = {k: v for k, v in mcmc.get_samples().items() if k in LATENT_SITES}
    return forecast(
        k_fc, model, posterior, train_data, full_covariates, batch_size=batch_size
    )


backtest_results = {}
for mode in MODES:
    rng_key, rng_subkey = random.split(rng_key)
    t_start = time.time()
    backtest_results[mode] = backtest(
        rng_subkey,
        y,
        covariates,
        lambda mode=mode: make_model(mu, U, mode=mode, duration=DURATION),
        forecast_fn=forecast_fn,
        metrics={"crps": eval_crps, "rmse": eval_rmse, "coverage_90": eval_coverage},
        min_train_window=DURATION - H - 5 * 4,
        test_window=H,
        stride=4,
        num_samples=1_000,
    )
    n_windows = len(backtest_results[mode])
    print(f"{mode:12s} backtest ({n_windows} windows) in {time.time() - t_start:.0f} s")

In [ ]:
backtest_df = pl.DataFrame(
    [
        {"model": mode, "window": i, "t1": res.t1, **res.metrics}
        for mode, results in backtest_results.items()
        for i, res in enumerate(results)
    ]
)

backtest_summary = backtest_df.group_by("model", maintain_order=True).agg(
    pl.col("crps").mean().alias("crps_mean"),
    pl.col("rmse").mean().alias("rmse_mean"),
    pl.col("coverage_90").mean().alias("coverage_90_mean"),
)
backtest_summary

In [ ]:
fig, ax = plt.subplots()
for i, mode in enumerate(MODES):
    sel = backtest_df.filter(pl.col("model") == mode)
    ax.plot(sel["t1"], sel["crps"], marker="o", color=f"C{i}", label=mode)
ax.set(xlabel="forecast origin (week)", ylabel="CRPS")
ax.legend()
fig.suptitle("Backtest CRPS per window", fontsize=18, fontweight="bold");

The paired differences per window are the honest summary: the same window, the same data, only the prior changes.

In [ ]:
paired = (
    backtest_df.pivot(on="model", index="window", values="crps")
    .with_columns(
        (pl.col("kron_sum") - pl.col("separable")).alias("kron_sum - separable"),
        (pl.col("kron_sum") - pl.col("independent")).alias("kron_sum - independent"),
        (pl.col("separable") - pl.col("independent")).alias("separable - independent"),
    )
    .select("kron_sum - separable", "kron_sum - independent", "separable - independent")
)

paired.describe().filter(pl.col("statistic").is_in(["mean", "std", "min", "max"]))

## Conclusion

We built a non-separable Gaussian process on the product space *interval times graph* from the eigenpairs of a Kronecker-sum Laplacian. The construction keeps everything that makes the HSGP approximation practical: the basis is fixed and hyperparameter-free, the weights are diagonal, and the latent panel is two matrix multiplications. The modeling content is in one scalar, the pooling strength $\rho$: rough spatial patterns get a shorter temporal memory and a smaller amplitude than the national trend, and the graph decides which patterns are suppressed first.

Four checks come with the construction and we recommend all of them:

- Trace normalization fixes the scale, and node standardization removes the variance heterogeneity that an irregular graph induces. Without it, low-degree regions get much wider prior bands for no modeling reason.
- The basis size must resolve the *shortest* effective lengthscale $\ell / \sqrt{1 + \rho \mu_{\max}}$. Print it from the posterior. A too-small basis acts as extra pooling.
- The time box must be fixed from the full duration, not from the length of the current call, so that training and forecasting share the same basis.
- Compare against the separable and the independent priors with the same hyperparameters, and do it on a backtest. On this synthetic panel the Kronecker sum has the lowest CRPS in every window, and most of the gain comes from one mechanism: the sum lets the national mode keep a long lengthscale while the rough spatial modes get a short one, so the trend is extrapolated instead of being pulled back to the intercepts. The size of the gain depends on how much the trend moves near the forecast origin, and the data generating process is a diffusion on the same graph. If the gain does not show on real data, the honest conclusion is that the separable prior is enough.

The same structure extends to a periodic time Laplacian (a seasonality that is shared nationally but blurred regionally), to three factors (interval times region graph times product graph), and to a fractional graph exponent that decouples the spatial from the temporal smoothness. What does not keep the structure is learning the graph weights: the basis would then depend on parameters and the eigendecomposition would move inside the model.

## References

- Riutort-Mayol, G., Bürkner, P.-C., Andersen, M. R., Solin, A., and Vehtari, A. (2023). [Practical Hilbert space approximate Bayesian Gaussian processes for probabilistic programming](https://link.springer.com/article/10.1007/s11222-022-10167-2). *Statistics and Computing*, 33, 17.
- Solin, A. and Särkkä, S. (2020). [Hilbert space methods for reduced-rank Gaussian process regression](https://link.springer.com/article/10.1007/s11222-019-09886-w). *Statistics and Computing*, 30, 419-446.
- Rasmussen, C. E. and Williams, C. K. I. (2006). [Gaussian Processes for Machine Learning, chapter 4: Covariance Functions](https://gaussianprocess.org/gpml/chapters/RW4.pdf). MIT Press.
- Borovitskiy, V., Azangulov, I., Terenin, A., Mostowsky, P., Deisenroth, M. P., and Durrande, N. (2021). [Matérn Gaussian Processes on Graphs](https://proceedings.mlr.press/v130/borovitskiy21a.html). *AISTATS 2021*.
- Engels, B., Andorra, A., and Kochurov, M. (2024). [Gaussian Processes: HSGP Advanced Usage](https://www.pymc.io/projects/examples/en/latest/gaussian_processes/HSGP-Advanced.html). PyMC examples.
- Saunders, D. (2023). [The Besag-York-Mollie Model for Spatial Data](https://www.pymc.io/projects/examples/en/latest/spatial/nyc_bym.html). PyMC examples.
- Orduz, J. (2024). [A Conceptual and Practical Introduction to Hilbert Space GPs Approximation Methods](https://juanitorduz.github.io/hsgp_intro/).
- Orduz, J. (2026). [CRPS](https://juanitorduz.github.io/crps/).
- Orduz, J. (2018). [Laplacian Eigenmaps for Dimensionality Reduction](https://juanitorduz.github.io/laplacian_eigenmaps_dim_red/) and the [PyData Berlin 2018 slides](https://juanitorduz.github.io/documents/orduz_pydata2018.pdf).
- [isellsoap/deutschlandGeoJSON](https://github.com/isellsoap/deutschlandGeoJSON): state polygons of Germany, derived from data of the Federal Agency for Cartography and Geodesy ([dl-de/by-2-0](https://www.govdata.de/dl-de/by-2-0)).
- [numpyro-forecast](https://github.com/juanitorduz/numpyro_forecast): forecasting building blocks, predictive helpers, and backtesting for NumPyro.
- [NumPyro HSGP module](https://num.pyro.ai/en/stable/contrib.html#hilbert-space-gaussian-processes-approximation): `numpyro.contrib.hsgp`.